In [ ]:
# imports
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from sympy import true
from sympy.codegen.ast import none

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")


In [ ]:
requests.get("http://localhost:11434/").content


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
ollama_url = "http://localhost:11434/v1"


anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)



In [ ]:
# Model names
ollama_model = 'llama3.2'
anthropic_model = 'claude-haiku-5-5'
gpt_model = 'gpt-6-luna'

In [ ]:
gpt_system = "You are Sam, a budget backpacker planning a 3-day trip to Kathmandu with two friends. \
You want to spend as little money as possible: cheap hostels, street food, local buses, walking. \
You push back when anyone suggests something expensive and offer a cheaper alternative. \
You are friendly and enthusiastic, but stubborn about money. \
Keep every reply to 2-3 sentences and respond directly to what the others just said."

claude_system = "You are Priya, a luxury traveler planning a 3-day trip to Kathmandu with two friends. \
You want comfort and special experiences: nice hotels, fine dining, private drivers, mountain flights. \
You think saving money is less important than making memories, and you try to convince the others. \
You are charming and a little dramatic. \
Keep every reply to 2-3 sentences and respond directly to what the others just said."

ollama_system = "You are Bikash, a friendly local tour guide who has lived in Kathmandu all your life. \
You know the real city: hidden temples, the best momo places, festivals, and what tourists get wrong. \
Your job is to find a middle ground between the budget traveler and the luxury traveler, \
and to share local tips they would never find online. \
Keep every reply to 2-3 sentences."

gpt_messages = ["Guys, I found a hostel in Thamel for just $6 a night, and we can take the local bus everywhere. Let's keep this whole trip under $100!"]
claude_messages = ["Under $100? Sam, darling, I've already looked at Dwarika's Hotel, and I want to see Everest from a mountain flight. We only do this once!"]
ollama_messages = ["Namaste, friends! You're both forgetting the best part of Kathmandu: the old temples and the momo shops, and those don't need a big budget or a fancy hotel. Let me show you the real city."]

In [ ]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, claude, llama in zip(gpt_messages, claude_messages, ollama_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
        messages.append({"role": "user", "content": llama})
    response = openai.chat.completions.create(model=gpt_model, reasoning_effort="none", messages=messages)
    return response.choices[0].message.content

In [ ]:
call_gpt()

In [ ]:
def call_claude():
    messages = [{"role": "system", "content": claude_system}]
    for gpt, claude_msg, llama in zip(gpt_messages, claude_messages, ollama_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude_msg})
        messages.append({"role": "user", "content": llama})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = anthropic.chat.completions.create(model=anthropic_model,  messages=messages, max_tokens=500)
    return response.choices[0].message.content

In [ ]:
def call_ollama():
    messages = [{"role": "system", "content": ollama_system}]
    for gpt, claude_msg, llama in zip(gpt_messages, claude_messages, ollama_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "user", "content": claude_msg})
        messages.append({"role": "assistant", "content": llama})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    messages.append({"role": "user", "content": claude_messages[-1]})
    response = ollama.chat.completions.create(model=ollama_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_claude()

In [ ]:
call_ollama()

In [ ]:
opening = [
    {"speaker": "Sam", "text": gpt_messages[0]},
    {"speaker": "Priya", "text": claude_messages[0]},
    {"speaker": "Bikash", "text": ollama_messages[0]},
]
conversation = list(opening)

In [ ]:
print(conversation[1]["speaker"])
print(conversation[1]["text"])


In [ ]:
def format_conversation(conversation):
    lines = []
    for turn in conversation:
        lines.append(f"{turn['speaker']}: {turn['text']}")
    return "\n\n".join(lines)

In [ ]:
print(format_conversation(conversation))

In [ ]:
from IPython.core.display_functions import update_display


def speak(client, model, system_prompt, name,
          instruction="Now respond with what you would like to say next", heading= None):
    script = format_conversation(conversation)

    user_prompt = f"""
You are {name}, planning a 3-day trip to Kathmandu with your friends.
The conversation so far is as follows:

{script}

{instruction}, as {name}.
Reply with only {name}'s words: no "{name}:" prefix, and don't write lines for anyone else.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    title = heading or f"### {name}:"
    handle = display(Markdown(title), display_id=True)

    response = client.chat.completions.create(model=model, messages=messages, stream = True)
    reply = ""
    for chunk in response:
        reply += chunk.choices[0].delta.content or ""
        update_display(Markdown(f"{title}\n{reply}"), display_id=handle.display_id)
    return reply

In [ ]:
speak(openai, gpt_model, gpt_system, "Sam")

In [ ]:
speak(anthropic, anthropic_model, claude_system, "Priya")

In [ ]:
speak(ollama, ollama_model, ollama_system, "Bikash")

In [ ]:
conversation = list(opening)
for turn in conversation:
    display(Markdown(f"### {turn['speaker']}:\n{turn['text']}"))

for i in range(3):
    sam_reply = speak(openai, gpt_model, gpt_system, "Sam")
    conversation.append({"speaker": "Sam", "text": sam_reply})

    priya_reply = speak(anthropic, anthropic_model, claude_system, "Priya")
    conversation.append({"speaker": "Priya", "text": priya_reply})

    bikash_reply = speak(ollama, ollama_model, ollama_system, "Bikash")
    conversation.append({"speaker": "Bikash", "text": bikash_reply})

final_plan = speak(
    ollama, ollama_model, ollama_system, "Bikash",
    instruction="The discussion is over. Summarise the final 3-day itinerary that everyone can agree on, "
                "with Day 1, Day 2 and Day 3 as a short list, and mention one compromise for Sam and one for Priya",
    heading="### 🗺️ Bikash's Final Plan:"
)
conversation.append({"speaker": "Bikash", "text": final_plan})